# Time Series Analysis — Complete Notebook

This notebook provides a reusable workflow for loading, cleaning, imputing missing values, resampling, checking stationarity, decomposing a time series into trend/seasonality/residual components, and applying smoothing methods.

**Before running:** change `FILE_PATH`, `DATE_COLUMN`, and `VALUE_COLUMN` in the configuration cell. The code also supports automatic detection of a date column when `DATE_COLUMN = None`.

In [ ]:
# =========================
# 1. INSTALL / IMPORT
# =========================
# If needed, uncomment this line once:
# !pip install pandas numpy matplotlib seaborn statsmodels scikit-learn openpyxl

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')


In [ ]:
# =========================
# 2. CONFIGURATION
# =========================

FILE_PATH = 'your_dataset.csv'   # CSV, XLSX, or XLS
DATE_COLUMN = None               # Example: 'Date'. Use None for automatic detection.
VALUE_COLUMN = None              # Example: 'Sales'. Use None to select the first numeric column.

# Optional settings
FREQUENCY = None                 # Examples: 'D', 'W', 'MS', 'ME', 'QS', 'YS'. None = infer when possible.
SEASONAL_PERIOD = None            # Examples: 7=daily data with weekly seasonality, 12=monthly data with yearly seasonality.
FILL_METHOD = 'interpolate'       # 'interpolate', 'ffill', 'bfill', or 'mean'
ROLLING_WINDOW = 7                # Used for moving-average smoothing

print('Update FILE_PATH, DATE_COLUMN, and VALUE_COLUMN if automatic detection is not suitable.')


In [ ]:
# =========================
# 3. LOAD DATA + BASIC CLEANING
# =========================

def load_dataset(path):
    path = str(path)
    if path.lower().endswith('.csv'):
        return pd.read_csv(path)
    if path.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    raise ValueError('Supported formats: CSV, XLSX, XLS')

df = load_dataset(FILE_PATH)

print('Original shape:', df.shape)
display(df.head())

# Remove completely empty rows/columns and duplicate rows
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all').drop_duplicates()
   
   # Automatic date-column detection
if DATE_COLUMN is None:
    candidates = []
    for col in df.columns:
        parsed = pd.to_datetime(df[col], errors='coerce')
        success_rate = parsed.notna().mean()
        if success_rate >= 0.70:
            candidates.append((col, success_rate))
    if not candidates:
        raise ValueError('No suitable date column found. Set DATE_COLUMN manually.')
    DATE_COLUMN = max(candidates, key=lambda x: x[1])[0]

df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors='coerce')
df = df.dropna(subset=[DATE_COLUMN])

# Automatic value-column detection
if VALUE_COLUMN is None:
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        raise ValueError('No numeric value column found. Set VALUE_COLUMN manually.')
    VALUE_COLUMN = numeric_cols[0]

# Convert selected value column to numeric; invalid entries become NaN
df[VALUE_COLUMN] = pd.to_numeric(df[VALUE_COLUMN], errors='coerce')
df = df[[DATE_COLUMN, VALUE_COLUMN]].copy()
df = df.sort_values(DATE_COLUMN).drop_duplicates(subset=DATE_COLUMN, keep='last')
   
print('Date column:', DATE_COLUMN)
print('Value column:', VALUE_COLUMN)
print('Cleaned shape:', df.shape)
display(df.head())


In [ ]:
# =========================
# 4. SET DATETIME INDEX + CHECK MISSING VALUES
# =========================

ts = df.set_index(DATE_COLUMN)[VALUE_COLUMN].sort_index().astype(float)

print('Start:', ts.index.min())
print('End:', ts.index.max())
print('Number of observations:', len(ts))
print('Missing values:', ts.isna().sum())
print('Missing percentage:', round(ts.isna().mean() * 100, 2), '%')

plt.figure(figsize=(14, 5))
plt.plot(ts.index, ts.values)
plt.title('Original Time Series')
plt.xlabel('Date')
plt.ylabel(VALUE_COLUMN)
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 5. RESAMPLE TO A REGULAR TIME INTERVAL
# =========================

# Time-series decomposition and many forecasting methods work best with a regular frequency.
# If FREQUENCY is None, pandas tries to infer the frequency.
inferred_freq = pd.infer_freq(ts.dropna().index) if len(ts.dropna()) >= 3 else None
freq = FREQUENCY or inferred_freq

print('Inferred frequency:', inferred_freq)

if freq is not None:
    ts = ts.resample(freq).mean()
    print('Using frequency:', freq)
else:
    print('Could not infer a regular frequency. Set FREQUENCY manually if needed.')

print('Missing values after resampling:', ts.isna().sum())


In [ ]:
# =========================
# 6. FILL MISSING VALUES
# =========================

before_missing = ts.isna().sum()

if FILL_METHOD == 'interpolate':
    ts_clean = ts.interpolate(method='time').ffill().bfill()
elif FILL_METHOD == 'ffill':
    ts_clean = ts.ffill().bfill()
elif FILL_METHOD == 'bfill':
    ts_clean = ts.bfill().ffill()
elif FILL_METHOD == 'mean':
    ts_clean = ts.fillna(ts.mean())
else:
    raise ValueError("FILL_METHOD must be 'interpolate', 'ffill', 'bfill', or 'mean'")

after_missing = ts_clean.isna().sum()
print('Missing before:', before_missing)
print('Missing after:', after_missing)

plt.figure(figsize=(14, 5))
plt.plot(ts.index, ts, label='Before filling', alpha=0.6)
plt.plot(ts_clean.index, ts_clean, label='After filling', linewidth=1.5)
plt.title('Missing-Value Treatment')
plt.xlabel('Date')
plt.ylabel(VALUE_COLUMN)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 7. DESCRIPTIVE TIME-SERIES PLOTS
# =========================

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(ts_clean.index, ts_clean)
ax.set_title('Cleaned Time Series')
ax.set_xlabel('Date')
ax.set_ylabel(VALUE_COLUMN)
plt.tight_layout()
plt.show()

# Rolling mean and rolling standard deviation
rolling_mean = ts_clean.rolling(ROLLING_WINDOW).mean()
rolling_std = ts_clean.rolling(ROLLING_WINDOW).std()

plt.figure(figsize=(15, 6))
plt.plot(ts_clean, label='Original')
plt.plot(rolling_mean, label=f'{ROLLING_WINDOW}-period rolling mean')
plt.plot(rolling_std, label=f'{ROLLING_WINDOW}-period rolling std')
plt.title('Rolling Statistics')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 8. TIME-SERIES COMPONENTS: TREND, SEASONALITY, RESIDUAL
# =========================

period = SEASONAL_PERIOD
if period is None:
    # Common defaults; choose manually when your domain has a known cycle.
    if freq in ['D', 'B']:
        period = 7
    elif freq in ['W', 'W-SUN', 'W-MON']:
        period = 52
    elif freq in ['MS', 'M', 'ME']:
        period = 12
    elif freq in ['QS', 'Q', 'QE']:
        period = 4
    else:
        period = 12

print('Seasonal period used:', period)

if len(ts_clean.dropna()) >= 2 * period:
    decomposition = seasonal_decompose(ts_clean, model='additive', period=period, extrapolate_trend='freq')
    decomposition.plot(figsize=(15, 10))
    plt.suptitle('Time-Series Decomposition: Observed, Trend, Seasonal, Residual', y=1.02)
    plt.tight_layout()
    plt.show()

    components = pd.DataFrame({
        'Observed': decomposition.observed,
        'Trend': decomposition.trend,
        'Seasonality': decomposition.seasonal,
        'Irregular_Residual': decomposition.resid
    })
    display(components.head())
else:
    print(f'Need at least {2 * period} observations for decomposition; current data has {len(ts_clean)}.')


In [ ]:
# =========================
# 9. INDIVIDUAL COMPONENT PLOTS
# =========================

if 'decomposition' in globals():
    component_names = {
        'Trend': decomposition.trend,
        'Seasonality': decomposition.seasonal,
        'Irregular / Residual': decomposition.resid
    }

    for name, series in component_names.items():
        plt.figure(figsize=(15, 4))
        plt.plot(series)
        plt.title(name)
        plt.xlabel('Date')
        plt.ylabel('Component value')
        plt.tight_layout()
        plt.show()

    # Seasonal pattern by position within the period
    seasonal_pattern = decomposition.seasonal.groupby(np.arange(len(decomposition.seasonal)) % period).mean()
    plt.figure(figsize=(12, 4))
    plt.plot(range(period), seasonal_pattern.values, marker='o')
    plt.title('Average Seasonal Pattern')
    plt.xlabel(f'Position within {period}-period cycle')
    plt.ylabel('Seasonal effect')
    plt.tight_layout()
    plt.show()


In [ ]:
# =========================
# 10. CYCLIC / LONG-WAVE MOVEMENT
# =========================

# Important: classical decomposition separates trend and seasonality, but a true
# economic/business cycle is not automatically identifiable from a single series.
# This plot shows long-wave movement using a wider rolling mean as a practical diagnostic.

cycle_window = max(period * 2, ROLLING_WINDOW * 3)
cycle = ts_clean.rolling(cycle_window, center=True, min_periods=max(2, cycle_window // 2)).mean()

plt.figure(figsize=(15, 6))
plt.plot(ts_clean, alpha=0.45, label='Observed')
plt.plot(cycle, linewidth=2, label=f'Long-wave rolling mean ({cycle_window})')
plt.title('Cyclic / Long-Wave Movement Diagnostic')
plt.xlabel('Date')
plt.ylabel(VALUE_COLUMN)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 11. SMOOTHING METHODS
# =========================

# A) Simple Moving Average
sma = ts_clean.rolling(window=ROLLING_WINDOW, center=True).mean()

# B) Exponentially Weighted Moving Average
ewma = ts_clean.ewm(span=ROLLING_WINDOW, adjust=False).mean()

# C) Simple Exponential Smoothing
ses = SimpleExpSmoothing(ts_clean, initialization_method='estimated').fit().fittedvalues

# D) Holt's linear trend smoothing
holt = Holt(ts_clean, initialization_method='estimated').fit(optimized=True).fittedvalues

plt.figure(figsize=(15, 7))
plt.plot(ts_clean, label='Original', alpha=0.45)
plt.plot(sma, label='Simple Moving Average')
plt.plot(ewma, label='EWMA')
plt.plot(ses, label='Simple Exponential Smoothing')
plt.plot(holt, label="Holt's Linear Smoothing")
plt.title('Comparison of Smoothing Methods')
plt.xlabel('Date')
plt.ylabel(VALUE_COLUMN)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 12. STL DECOMPOSITION — MORE ROBUST OPTION
# =========================

if len(ts_clean.dropna()) >= 2 * period:
    stl = STL(ts_clean, period=period, robust=True).fit()
    fig = stl.plot()
    fig.set_size_inches(15, 10)
    plt.suptitle('STL Decomposition', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# =========================
# 13. STATIONARITY TEST (ADF)
# =========================

def adf_test(series, name='Series'):
    series = series.dropna()
    result = adfuller(series, autolag='AIC')
    print(f'ADF test: {name}')
    print('ADF statistic:', result[0])
    print('p-value:', result[1])
    print('Used lags:', result[2])
    print('Observations:', result[3])
    print('Conclusion:', 'Likely stationary (p < 0.05)' if result[1] < 0.05 else 'Likely non-stationary (p >= 0.05)')

adf_test(ts_clean, 'Original cleaned series')

# First difference
ts_diff = ts_clean.diff().dropna()
adf_test(ts_diff, 'First-differenced series')

plt.figure(figsize=(15, 5))
plt.plot(ts_diff)
plt.title('First Difference')
plt.xlabel('Date')
plt.ylabel('Differenced value')
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 14. ACF AND PACF
# =========================

lags = min(40, max(1, len(ts_clean)//3))

fig, ax = plt.subplots(figsize=(14, 5))
plot_acf(ts_clean.dropna(), lags=lags, ax=ax)
ax.set_title('Autocorrelation Function (ACF)')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
plot_pacf(ts_clean.dropna(), lags=lags, ax=ax, method='ywm')
ax.set_title('Partial Autocorrelation Function (PACF)')
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 15. OPTIONAL: LOG TRANSFORMATION
# =========================

# Useful when variance increases with the level of the series.
# Only valid for strictly positive values.

if (ts_clean > 0).all():
    log_ts = np.log(ts_clean)
    plt.figure(figsize=(15, 5))
    plt.plot(log_ts)
    plt.title('Log-Transformed Series')
    plt.xlabel('Date')
    plt.ylabel('log(' + VALUE_COLUMN + ')')
    plt.tight_layout()
    plt.show()
else:
    print('Log transformation skipped because the series contains zero or negative values.')


In [ ]:
# =========================
# 16. SAVE THE PROCESSED DATA
# =========================

output = pd.DataFrame({
    DATE_COLUMN: ts_clean.index,
    VALUE_COLUMN: ts_clean.values
})

if 'decomposition' in globals():
    output['Trend'] = decomposition.trend.values
    output['Seasonality'] = decomposition.seasonal.values
    output['Irregular_Residual'] = decomposition.resid.values

output['SMA'] = sma.values
output['EWMA'] = ewma.values

output.to_csv('processed_time_series.csv', index=False)
display(output.head(10))
print('Saved: processed_time_series.csv')
